<a href="https://colab.research.google.com/github/Zuhair0000/TensorFlow-ML-DL-Project-Practice/blob/main/practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
import pandas as pd
import numpy as np
from sklearn.datasets import make_classification

# Generate a synthetic imbalanced dataset (like fraud)
X, y = make_classification(n_samples=10000, n_features=10, n_classes=2, weights=[0.95, 0.05], random_state=42)
df = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(10)])
df['target'] = y

# Inject some fake missing values (-1) to simulate real-world messy data
df.loc[np.random.choice(df.index, 500), 'feature_3'] = -1
df.to_csv('dataset.csv', index=False)
print("dataset.csv created successfully!")

dataset.csv created successfully!


In [30]:
import pandas as pd
import numpy as np


df = pd.read_csv('dataset.csv')

In [31]:
df['feature_3'] = df['feature_3'].replace(-1, np.nan)

In [32]:
X = df.drop(columns=['target'])
y = df['target']

In [33]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.8, random_state=42, stratify=y)

In [34]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_cols = X_train.columns.to_list()

num_enc = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer([
    ('num', num_enc, num_cols)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# ML

In [35]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=50, class_weight='balanced', random_state=42)
rf.fit(X_train_processed, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=50,
                       random_state=42)

In [36]:
predictions = rf.predict(X_test_processed)
predictions

array([0, 0, 1, ..., 0, 0, 0])

In [37]:
from sklearn.metrics import classification_report

# DL (TF)

In [38]:
import tensorflow as tf

tf_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_processed.shape[1], )),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

tf_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss = 'binary_crossentropy'
                 )

tf_model.fit(X_train_processed, y_train, epochs=5, batch_size=64)

Epoch 1/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.3100
Epoch 2/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1687
Epoch 3/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.1507
Epoch 4/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.1427
Epoch 5/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.1343


In [39]:
tf_outputs = tf_model.predict(X_test_processed)
tf_predictions = (tf_outputs > 0.5).astype(int)

print(classification_report(y_test, tf_predictions))

250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
              precision    recall  f1-score   support

           0       0.96      0.99      0.98      7558
           1       0.71      0.24      0.36       442

    accuracy                           0.95      8000
   macro avg       0.84      0.62      0.67      8000
weighted avg       0.94      0.95      0.94      8000



# DL (Pytorch)

In [43]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

X_train_tensor = torch.tensor(X_train_processed, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_processed, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1, 1)

In [46]:
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=64, shuffle=False)

In [48]:
class FraudNet(nn.Module):
  def __init__(self, input_dim):
    super(FraudNet, self).__init__()
    self.layer1 = nn.Linear(input_dim, 16)
    self.relu = nn.ReLU()
    self.layer2 = nn.Linear(16, 1)

  def forward(self, x):
    x = self.relu(self.layer1(x))
    x = self.layer2(x)
    return x

In [53]:
model = FraudNet(input_dim= X_train_processed.shape[1])
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

epochs = 5

for epoch in range(epochs):
  model.train()
  for batch_X, batch_y in train_loader:
    optimizer.zero_grad()
    outputs = model(batch_X)
    loss = criterion(outputs, batch_y)
    loss.backward()
    optimizer.step()

In [56]:
model.eval()

with torch.no_grad():
  test_outputs = model(X_test_tensor)
  pt_predicitions = (torch.sigmoid(test_outputs)>0.5).float()

print(classification_report(y_test_tensor, pt_predicitions))

              precision    recall  f1-score   support

         0.0       0.96      0.99      0.98      7558
         1.0       0.68      0.25      0.37       442

    accuracy                           0.95      8000
   macro avg       0.82      0.62      0.67      8000
weighted avg       0.94      0.95      0.94      8000

